In [97]:
from pyspark.sql import SparkSession
import getpass 
username=getpass.getuser()
spark=SparkSession. \
    builder. \
    config('spark.ui.port','0'). \
    config("spark.sql.warehouse.dir", f"/user/{username}/warehouse"). \
    config('spark.shuffle.useOldFetchProtocol', 'true'). \
    enableHiveSupport(). \
    master('yarn'). \
    getOrCreate()

In [98]:
spark

In [130]:
orders_rdd = spark.sparkContext.textFile("/public/trendytech/retail_db/orders/*")

In [131]:
orders_rdd.take(10)

['1,2013-07-25 00:00:00.0,11599,CLOSED',
 '2,2013-07-25 00:00:00.0,256,PENDING_PAYMENT',
 '3,2013-07-25 00:00:00.0,12111,COMPLETE',
 '4,2013-07-25 00:00:00.0,8827,CLOSED',
 '5,2013-07-25 00:00:00.0,11318,COMPLETE',
 '6,2013-07-25 00:00:00.0,7130,COMPLETE',
 '7,2013-07-25 00:00:00.0,4530,COMPLETE',
 '8,2013-07-25 00:00:00.0,2911,PROCESSING',
 '9,2013-07-25 00:00:00.0,5657,PENDING_PAYMENT',
 '10,2013-07-25 00:00:00.0,5648,PENDING_PAYMENT']

In [132]:
mapped_rdd = orders_rdd.map(lambda x: (x.split(",")[3],1))

In [133]:
mapped_rdd.take(5)

[('CLOSED', 1),
 ('PENDING_PAYMENT', 1),
 ('COMPLETE', 1),
 ('CLOSED', 1),
 ('COMPLETE', 1)]

In [134]:
reduced_rdd = mapped_rdd.reduceByKey(lambda x,y:x+y)

In [135]:
reduced_rdd.collect()

[('CLOSED', 7556),
 ('CANCELED', 1428),
 ('PENDING_PAYMENT', 15030),
 ('COMPLETE', 22899),
 ('PROCESSING', 8275),
 ('PAYMENT_REVIEW', 729),
 ('PENDING', 7610),
 ('ON_HOLD', 3798),
 ('SUSPECTED_FRAUD', 1558)]

In [136]:
customer_rdd = orders_rdd.map(lambda x: (x.split(",")[2],1))

In [137]:
customer_rdd.take(10)

[('11599', 1),
 ('256', 1),
 ('12111', 1),
 ('8827', 1),
 ('11318', 1),
 ('7130', 1),
 ('4530', 1),
 ('2911', 1),
 ('5657', 1),
 ('5648', 1)]

In [138]:
customer_reduced_rdd = customer_rdd.reduceByKey(lambda x,y:x+y)

In [139]:
customer_reduced_rdd.take(10)

[('3066', 6),
 ('3159', 7),
 ('8135', 11),
 ('2248', 4),
 ('6117', 6),
 ('7733', 7),
 ('6540', 3),
 ('4882', 8),
 ('6060', 7),
 ('10436', 8)]

In [141]:
sorted_rdd = customer_reduced_rdd.sortBy(lambda x:x[1], False)

In [142]:
sorted_rdd.take(10)

[('6316', 16),
 ('12431', 16),
 ('569', 16),
 ('5897', 16),
 ('5283', 15),
 ('12284', 15),
 ('5654', 15),
 ('221', 15),
 ('4320', 15),
 ('5624', 15)]

In [111]:
# distinct count of customers who placed atleast one order

distinct_customers_rdd = orders_rdd.map(lambda x: (x.split(",")[2],1)).distinct()

In [112]:
distinct_customers_rdd.count()

12405

In [113]:
orders_rdd.count()

68883

In [143]:
#which customers has the maximum number of CLOSED orders

closed_orders_rdd = orders_rdd.filter(lambda x:(x.split(",")[3]) == 'CLOSED')

In [144]:
closed_orders_rdd.take(10)

['1,2013-07-25 00:00:00.0,11599,CLOSED',
 '4,2013-07-25 00:00:00.0,8827,CLOSED',
 '12,2013-07-25 00:00:00.0,1837,CLOSED',
 '18,2013-07-25 00:00:00.0,1205,CLOSED',
 '24,2013-07-25 00:00:00.0,11441,CLOSED',
 '25,2013-07-25 00:00:00.0,9503,CLOSED',
 '37,2013-07-25 00:00:00.0,5863,CLOSED',
 '51,2013-07-25 00:00:00.0,12271,CLOSED',
 '57,2013-07-25 00:00:00.0,7073,CLOSED',
 '61,2013-07-25 00:00:00.0,4791,CLOSED']

In [145]:
mapped_orders = closed_orders_rdd.map(lambda x:(x.split(",")[2], 1))

In [146]:
mapped_orders.take(10)

[('11599', 1),
 ('8827', 1),
 ('1837', 1),
 ('1205', 1),
 ('11441', 1),
 ('9503', 1),
 ('5863', 1),
 ('12271', 1),
 ('7073', 1),
 ('4791', 1)]

In [147]:
reduced_orders = mapped_orders.reduceByKey(lambda x, y: x+y)

In [148]:
reduced_orders.take(10)

[('3159', 1),
 ('5834', 2),
 ('10173', 1),
 ('2101', 1),
 ('6000', 1),
 ('1352', 2),
 ('10142', 1),
 ('12210', 1),
 ('6018', 2),
 ('2252', 1)]

In [151]:
sorted_custoemrs = reduced_orders.sortBy(lambda x:x[1], False)

In [152]:
sorted_custoemrs.take(10)

[('1833', 6),
 ('5493', 5),
 ('1687', 5),
 ('1363', 5),
 ('9740', 4),
 ('4596', 4),
 ('2430', 4),
 ('11833', 4),
 ('9830', 4),
 ('9804', 4)]